## Complete RAG pipeline — URL, PDF, DOCX, or text/markdown input

In [1]:
import os
from pathlib import Path

from dotenv import load_dotenv
from langchain_community.document_loaders import (
    Docx2txtLoader,
    PyPDFLoader,
    TextLoader,
    WebBaseLoader,
)
from langchain_chroma import Chroma
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv()

os.environ.setdefault("USER_AGENT", "generative-ai-projects-rag/1.0")

USER_AGENT environment variable not set, consider setting it to identify your requests.


'generative-ai-projects-rag/1.0'

### 1. Load — detect input type (URL vs file extension) and pick the right loader

In [2]:
def load_documents(source: str):
    """source: a URL, or a path to a .pdf / .docx / .txt / .md file."""
    if source.startswith("http://") or source.startswith("https://"):
        loader = WebBaseLoader(source)
    else:
        suffix = Path(source).suffix.lower()
        if suffix == ".pdf":
            loader = PyPDFLoader(source)
        elif suffix == ".docx":
            loader = Docx2txtLoader(source)
        elif suffix in (".txt", ".md"):
            loader = TextLoader(source)
        else:
            raise ValueError(f"Unsupported source: {source}")
    return loader.load()

### 2. Split → 3. Embed → 4. Store in Chroma

In [3]:
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")


def build_vectorstore(source: str, persist_directory: str = "chroma_db"):
    docs = load_documents(source)

    splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
    chunks = splitter.split_documents(docs)

    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        persist_directory=persist_directory,
    )
    return vectorstore

### 5. Retrieve → 6. Generate answer with Gemini

In [4]:
llm = ChatGoogleGenerativeAI(model="gemini-3-flash-preview")

prompt = ChatPromptTemplate.from_template(
    "Answer the question using only the context below. "
    "If the answer isn't in the context, say you don't know.\n\n"
    "Context:\n{context}\n\nQuestion: {question}"
)


def _format_docs(docs) -> str:
    return "\n\n".join(doc.page_content for doc in docs)


def _extract_text(response) -> str:
    if isinstance(response.content, str):
        return response.content
    return "".join(
        block["text"] for block in response.content if isinstance(block, dict) and block.get("type") == "text"
    )


def ask(vectorstore, question: str, k: int = 4) -> str:
    retriever = vectorstore.as_retriever(search_kwargs={"k": k})
    docs = retriever.invoke(question)
    messages = prompt.invoke({"context": _format_docs(docs), "question": question})
    response = llm.invoke(messages)
    return _extract_text(response)

### Try it — swap `source` for a URL, a `.pdf`, a `.docx`, or a `.txt`/`.md` path

In [5]:
source = "https://lilianweng.github.io/posts/2023-06-23-agent/"
vectorstore = build_vectorstore(source)

In [6]:
print(ask(vectorstore, "What are the main components of an LLM-powered autonomous agent?"))

According to the context, the main components of an LLM-powered autonomous agent system are:

*   **LLM (Large Language Model):** Functions as the agent's "brain" or core controller.
*   **Planning:** Involves task decomposition (breaking down large tasks into subgoals) and reflection/refinement (self-criticism and learning from mistakes).
*   **Memory:** Includes short-term memory (in-context learning) and long-term memory (retaining and recalling information via an external vector store).
*   **Tool use:** The capability to call external APIs for extra information, code execution, or access to proprietary sources.
